# Pseudo-labeling 시도

**먼저 규칙 확인이 필요합니다.** DACON 규칙 페이지에 명시된 유의사항:

> 모델 학습에서 평가 데이터셋 활용(Data Leakage)시 수상 제외

Pseudo-labeling은 test 데이터의 예측 결과(라벨이 아니라 모델이 만든 예측)를
다시 학습에 사용하는 기법입니다. 이게 "평가 데이터셋 활용"에 해당하는지
애매합니다 — 실제 정답 라벨을 쓰는 게 아니라 모델 자신의 예측을 쓰는
거라 일반적인 ML 대회에서는 허용되는 경우가 많지만, **이 대회 규칙에
명시적으로 허용 여부가 적혀있지 않습니다.**

**권장: 이 노트북을 실행하기 전에 대회 운영진(질문 게시판 등)에 직접
문의해서 pseudo-labeling이 규칙상 허용되는지 확인하세요.** 확인 전에는
실행하지 않거나, 실행하더라도 실제 제출에는 사용하지 않는 것이 안전합니다.

## 기법 설명

1. 현재 main.py 파이프라인으로 5-Fold 모델을 학습
2. test 데이터에 대한 예측 확률 생성
3. 확신도가 매우 높은 예측만 선별 (예: 0.03 이하 또는 0.97 이상)
4. 그 test 행들을 "가짜 라벨"(0 또는 1)과 함께 학습 데이터에 추가
5. 확장된 데이터로 다시 5-Fold 학습 -> 검증 점수 확인


## 1. 라이브러리 및 설정

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
TARGET_COL = "임신 성공 여부"
ID_COL = "ID"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

CONFIDENCE_THRESHOLD_LOW = 0.03   # 이 값 이하면 "확실한 실패"로 간주
CONFIDENCE_THRESHOLD_HIGH = 0.97  # 이 값 이상이면 "확실한 성공"으로 간주

## 2. 전처리 (main.py와 동일)

In [ ]:
def preprocess(train, test):
    test_ids = test[ID_COL].copy()
    train = train.drop(columns=[ID_COL])
    test = test.drop(columns=[ID_COL])

    for df in (train, test):
        df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
        df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)

    train = train.drop(columns=DEAD_COLS)
    test = test.drop(columns=DEAD_COLS)

    categorical_cols = train.select_dtypes(include=["object", "str"]).columns
    numerical_cols = train.select_dtypes(include=["int64", "float64"]).columns
    na_cols = train.isna().sum().loc[lambda x: x > 0].index

    for col in na_cols:
        if col in categorical_cols:
            train[col] = train[col].fillna("해당없음")
            test[col] = test[col].fillna("해당없음")
        elif col in numerical_cols:
            train[col] = train[col].fillna(-1)
            test[col] = test[col].fillna(-1)

    for col in categorical_cols:
        le = LabelEncoder()
        le.fit(train[col])
        unseen_labels = set(test[col].unique()) - set(le.classes_)
        if unseen_labels:
            le.classes_ = np.append(le.classes_, list(unseen_labels))
        train[col] = le.transform(train[col])
        test[col] = le.transform(test[col])

    return train, test, test_ids


train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train_p, test_p, test_ids = preprocess(train_raw, test_raw)

X = train_p.drop(columns=[TARGET_COL])
y = train_p[TARGET_COL]
print(f"전처리 완료: train={X.shape}, test={test_p.shape}")

## 3. 베이스라인 모델 학습 (pseudo-label 생성용)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1004)
base_models = []
base_fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    model = LGBMClassifier(random_state=1004, verbose=-1)
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    proba = model.predict_proba(X.iloc[val_idx])[:, 1]
    score = roc_auc_score(y.iloc[val_idx], proba)
    base_fold_scores.append(score)
    base_models.append(model)
    print(f"[Fold {fold}/5] baseline ROC-AUC: {score:.4f}")

baseline_score = np.mean(base_fold_scores)
print(f"\n베이스라인 5-Fold 평균: {baseline_score:.5f}")

test_pred = np.mean([m.predict_proba(test_p)[:, 1] for m in base_models], axis=0)
print(f"test 예측 분포: min={test_pred.min():.4f}, max={test_pred.max():.4f}")

## 4. 확신도 높은 test 샘플 선별

In [ ]:
confident_low = test_pred <= CONFIDENCE_THRESHOLD_LOW
confident_high = test_pred >= CONFIDENCE_THRESHOLD_HIGH
confident_mask = confident_low | confident_high

n_confident = confident_mask.sum()
print(f"확신도 높은 test 샘플: {n_confident} / {len(test_pred)} ({n_confident/len(test_pred)*100:.1f}%)")
print(f"  - 확실한 실패(0)로 분류: {confident_low.sum()}건")
print(f"  - 확실한 성공(1)으로 분류: {confident_high.sum()}건")

pseudo_X = test_p[confident_mask].copy()
pseudo_y = (test_pred[confident_mask] >= 0.5).astype(int)
pseudo_y = pd.Series(pseudo_y, index=pseudo_X.index)

print(f"\npseudo-label 분포: {pd.Series(pseudo_y).value_counts().to_dict()}")

## 5. 확장된 데이터로 재학습 및 검증

In [ ]:
X_extended = pd.concat([X, pseudo_X], axis=0, ignore_index=True)
y_extended = pd.concat([y, pseudo_y], axis=0, ignore_index=True)

print(f"기존 train: {len(X)}건 -> 확장 후: {len(X_extended)}건 (+{len(pseudo_X)}건)")

# 주의: 검증은 원래 train 데이터에 대해서만 수행해야 공정한 비교가 됨
# (pseudo-label 데이터가 검증셋에 섞이면 결과가 부풀려짐)
skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=1004)
extended_fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf2.split(X, y), start=1):
    # 학습 데이터에는 pseudo-label을 추가하지만, 검증은 원본 train 데이터의 val_idx로만
    X_train_fold = pd.concat([X.iloc[tr_idx], pseudo_X], axis=0, ignore_index=True)
    y_train_fold = pd.concat([y.iloc[tr_idx], pseudo_y], axis=0, ignore_index=True)

    model = LGBMClassifier(random_state=1004, verbose=-1)
    model.fit(X_train_fold, y_train_fold)
    proba = model.predict_proba(X.iloc[val_idx])[:, 1]
    score = roc_auc_score(y.iloc[val_idx], proba)
    extended_fold_scores.append(score)
    print(f"[Fold {fold}/5] pseudo-label 포함 ROC-AUC: {score:.4f}")

extended_score = np.mean(extended_fold_scores)
print(f"\npseudo-label 포함 5-Fold 평균: {extended_score:.5f}")
print(f"베이스라인 5-Fold 평균: {baseline_score:.5f}")
print(f"차이: {extended_score - baseline_score:+.5f}")

## 6. 결론 및 주의사항

개선이 확인되더라도, **실제 제출 전에 반드시 대회 규칙(data leakage 조항)에
저촉되지 않는지 재확인하세요.** 애매하면 운영진에게 직접 문의하는 것이
가장 안전합니다. 규칙 위반으로 수상 제외되는 것보다, 약간 낮은 점수로
안전하게 제출하는 것이 낫습니다.
